# Israeli 26th Knesset results map

This is the 2026 version of the repository's interactive election map. It joins each ballot box to its polling-place coordinates and builds polling-place Voronoi regions clipped to the corresponding locality boundary. Hover for a quick summary; click a region for its party chart.

### Map data

The boundary layer follows the pipeline used by [RavBogard/israelielection](https://github.com/RavBogard/israelielection/blob/main/scripts/vote-map/build.mts): Ministry of Transport 2026 locality polygons, supplemented by dissolved CBS 2008 statistical areas for West Bank localities. Full source URLs and caveats are in [2026/MAP_SOURCES.md](2026/MAP_SOURCES.md).

In [1]:
%matplotlib widget
from math import cos, pi, sin

import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter
from shapely.geometry import MultiPoint, Point, mapping, shape
from shapely.ops import voronoi_diagram
from ipyleaflet import (
    FullScreenControl, GeoJSON, Map, ScaleControl,
    basemaps,
)
from ipywidgets import HBox, HTML, Label, Layout, VBox

from behirot_2026 import (
    download_election_file, download_polling_places, topojson_to_geojson,
)

## Refresh the official results

This downloads and caches the official ballot-box export and polling-place table under 2026/. Results and locations are joined by the `ברזל` identifier.

In [2]:
ballot_results = download_election_file("expb.csv")
polling_places = download_polling_places()

def clean_export(frame):
    frame = frame.loc[:, ~frame.columns.str.startswith("Unnamed:")].copy()
    numeric_columns = frame.columns.difference(["שם ישוב"])
    frame[numeric_columns] = frame[numeric_columns].apply(
        pd.to_numeric, errors="coerce"
    ).fillna(0)
    return frame

ballot_results = clean_export(ballot_results)
ballot_results["ברזל"] = ballot_results["ברזל"].astype("Int64").astype("string")

party_column_index = ballot_results.columns.get_loc("קלפי אחרת") + 1
party_columns = list(ballot_results.columns[party_column_index:])
location_columns = [
    "ברזל", "סמל קלפי", "כתובת קלפי", "מקום קלפי",
    "קו אורך", "קו רוחב",
]
ballot_results = ballot_results.merge(
    polling_places[location_columns],
    on="ברזל", how="left", validate="one_to_one",
    suffixes=("", "_מיקום"),
)
located_results = ballot_results.dropna(subset=["קו אורך", "קו רוחב"]).copy()
if len(located_results) != len(ballot_results):
    missing = len(ballot_results) - len(located_results)
    print(f"{missing} result rows have no polling-place coordinates")

## Build polling-place regions

In [3]:
locality_boundaries = topojson_to_geojson("2026/boundaries.topo.json")
boundary_by_code = {}
for feature in locality_boundaries["features"]:
    boundary = shape(feature["geometry"])
    if not boundary.is_valid:
        boundary = boundary.buffer(0)
    boundary_by_code[int(feature["properties"]["code"])] = boundary

def display_ballot_number(value):
    value = str(value)
    return value[:-2] if value.endswith(".0") else value

def ballot_properties(row):
    votes = row[party_columns].astype(int)
    dominant_party = votes.idxmax() if votes.sum() else ""
    address = " ".join(
        value for value in [row["כתובת קלפי"], row["מקום קלפי"]]
        if isinstance(value, str) and value.strip()
    )
    return {
        "name": row["שם ישוב"],
        "address": address,
        "number": display_ballot_number(row["קלפי"]),
        "iron_id": str(row["ברזל"]),
        "voted": int(row["מצביעים"]),
        "invalid": int(row["פסולים"]),
        "valid": int(row["כשרים"]),
        "votes": {party: int(votes[party]) for party in party_columns},
        "dominant_party": dominant_party,
    }

def separated_ballot_points(rows, distance=0.00002):
    ballot_points = []
    for (longitude, latitude), shared_rows in rows.groupby(
        ["קו אורך", "קו רוחב"], sort=False
    ):
        shared_rows = shared_rows.sort_values("קלפי")
        count = len(shared_rows)
        for index, (_, row) in enumerate(shared_rows.iterrows()):
            if count == 1:
                point = Point(float(longitude), float(latitude))
            else:
                angle = 2 * pi * index / count
                point = Point(
                    float(longitude) + distance * cos(angle),
                    float(latitude) + distance * sin(angle),
                )
            ballot_points.append((point, row))
    return ballot_points

polling_place_features = []
for locality_code, locality_rows in located_results.groupby("סמל ישוב"):
    boundary = boundary_by_code.get(int(locality_code))
    if boundary is None:
        continue

    ballots = separated_ballot_points(locality_rows)
    points = [point for point, _ in ballots]
    if len(points) == 1:
        regions = [boundary]
    else:
        diagram = voronoi_diagram(MultiPoint(points), envelope=boundary.envelope)
        cells = list(diagram.geoms)
        regions = [
            min(cells, key=lambda cell: cell.distance(point)).intersection(boundary)
            for point in points
        ]

    for (_, row), region in zip(ballots, regions):
        if region.is_empty:
            continue
        polling_place_features.append({
            "type": "Feature",
            "geometry": mapping(region),
            "properties": ballot_properties(row),
        })

polling_place_regions = {
    "type": "FeatureCollection",
    "features": polling_place_features,
}


## Interactive map

In [4]:
plt.ioff()
fig, party_ax = plt.subplots(1, 1)
plt.ion()

def plot_party_votes_df(party_votes, ax):
    ax.cla()
    plot_data = party_votes.sort_values().copy()
    total = plot_data.sum()
    if not total:
        return
    plot_data = plot_data * 100 / total
    plot_data.plot(kind='barh', ax=ax)
    ax.set_ylim([max(len(plot_data)-10, 0), len(plot_data)])
    ax.set_xlim([0,100])
    ax.xaxis.set_major_formatter(PercentFormatter())

title_widget = HTML()

def set_format(fmt):
    def set_in_fmt(widget, variable):
        widget.value = fmt % variable
    return set_in_fmt

set_title = set_format("<h1>%s</h1>")
set_voted = set_format('מצביעים: %s')
set_invalid = set_format('פסולים: %s, %.2f%%')
set_valid = set_format('כשרים: %s, %.2f%%')
set_dominant_party = set_format('מפלגה דומיננטית: %s')
set_dominant_party_votes = set_format('הצבעות למלגה הדומיננטית: %s, %.2f%%')

voted_label = Label(style={"font_size": "23px"})
invalid_label = Label(style={"font_size": "23px"})
valid_label = Label(style={"font_size": "23px"})
dominant_party_label = Label(style={"font_size": "23px", "font_weight": "bold"})
dominant_party_votes_label = Label(style={"font_size": "23px"})

set_voted(voted_label, '')
set_invalid(invalid_label, ('', 0))
set_valid(valid_label, ('', 0))
set_title(title_widget, "Hover to display quick summary")
set_dominant_party(dominant_party_label, '')
set_dominant_party_votes(dominant_party_votes_label, ('', 0))

def get_dominant_party(party):
    total = party.sum()
    if not total:
        return '', 0, 0
    name = party.idxmax()
    votes = party.loc[name]
    return name, votes, votes/total

def show_ballot(feature, plot=False):
    properties = feature["properties"]
    party = pd.Series(properties["votes"])
    title = f'{properties["name"]} — קלפי {properties["number"]}'
    if properties["address"]:
        title += f'<br><small>{properties["address"]}</small>'
    set_title(title_widget, title)
    voted = properties["voted"]
    invalid = properties["invalid"]
    valid = properties["valid"]
    set_voted(voted_label, voted)
    set_valid(valid_label, (valid, valid*100/voted if voted else 0))
    set_invalid(invalid_label, (invalid, invalid*100/voted if voted else 0))
    name, votes, percent = get_dominant_party(party)
    set_dominant_party(dominant_party_label, name)
    set_dominant_party_votes(dominant_party_votes_label, (votes, percent*100))
    if plot:
        plot_party_votes_df(party, party_ax)
        party_ax.figure.suptitle(f'{properties["name"]} — קלפי {properties["number"]}')
        party_ax.figure.canvas.draw_idle()

labels = VBox([title_widget, dominant_party_label, voted_label, valid_label, invalid_label, dominant_party_votes_label])

In [5]:
m = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=(31.5, 34.9),
    zoom=7,
    min_zoom=6,
    max_zoom=15,
    scroll_wheel_zoom=True,
    prefer_canvas=True,
    world_copy_jump=False,
    layout=Layout(height="760px", width="60%"),
)

party_colors = {
    "מחל": "#245b94",  # Likud
    "ב": "#00a6c8",    # Otzma Yehudit
    "ט": "#4dd0e1",    # Religious Zionism–Zehut
    "ני": "#b29bda",   # Noam
    "ך": "#597fbd",    # Amcha Yisrael
    "שס": "#3f444a",   # Shas
    "ג": "#111111",    # United Torah Judaism
    "רק": "#ce542d",   # Beyachad
    "דרך": "#ed9859",  # Yashar
    "אמת": "#a72f65", # The Democrats
    "ל": "#9e3826",    # Yisrael Beiteinu
    "כן": "#db7898",   # Blue and White
    "די": "#b48727",   # Reservists–Economic
    "ודם": "#267b44",  # Joint List
    "עם": "#80b667",   # Ra'am
}
party_fallback_color = "#8c939b"
party_names = {
    "מחל": "הליכוד",
    "ב": "עוצמה יהודית",
    "ט": "הציונות הדתית–זהות",
    "ני": "נעם",
    "ך": "עמך ישראל",
    "שס": "ש״ס",
    "ג": "יהדות התורה",
    "רק": "ביחד",
    "דרך": "ישר",
    "אמת": "הדמוקרטים",
    "ל": "ישראל ביתנו",
    "כן": "כחול לבן",
    "די": "המילואימניקים–הכלכלית",
    "ודם": "הרשימה המשותפת",
    "עם": "רע״ם",
}

def dominant_party_color(feature):
    return {
        'color': 'black',
        'fillColor': party_colors.get(
            feature["properties"].get("dominant_party"),
            party_fallback_color,
        ),
    }

geo_json = GeoJSON(
    data=polling_place_regions,
    style={
        'opacity': 1, 'dashArray': '9', 'fillOpacity': 0.5, 'weight': 1
    },
    hover_style={
        'color': 'white', 'dashArray': '0', 'fillOpacity': 0.9
    },
    style_callback=dominant_party_color,
)
geo_json.on_hover(lambda feature, **kwargs: show_ballot(feature))
geo_json.on_click(lambda feature, **kwargs: show_ballot(feature, plot=True))
m.add(geo_json)

city_lines = GeoJSON(
    data=locality_boundaries,
    style={
        'color': '#000000', 'weight': 2, 'opacity': 1,
        'fill': False, 'interactive': False,
    },
)
m.add(city_lines)

panel = VBox(
    [labels, fig.canvas],
    layout=Layout(width="40%", max_height="760px"),
)
m.add(ScaleControl(position="bottomleft", imperial=False))
m.add(FullScreenControl(position="topleft"))
map_with_panel = HBox(
    [m, panel],
    layout=Layout(width="90vw"),
);



In [6]:
map_with_panel

In [7]:
total_party_votes = ballot_results[party_columns].sum().astype(int)
total_party_votes = total_party_votes[total_party_votes > 0].sort_values()
total_ballots_cast = int(ballot_results["מצביעים"].sum())
total_valid_votes = int(ballot_results["כשרים"].sum())
party_percent = total_party_votes * 100 / total_valid_votes

plt.ioff()
fig_total, total_ax = plt.subplots(figsize=(10, max(6, len(party_percent) * 0.35)))
plt.ion()
plt.axvline(3.25, ls='--', c='k')
party_percent.plot(
    kind="barh",
    ax=total_ax,
    color=[party_colors.get(party, party_fallback_color) for party in party_percent.index],
)
total_ax.set_title(
    f"סה״כ מצביעים: {total_ballots_cast:,} | קולות כשרים: {total_valid_votes:,}"
)
total_ax.set_xlabel("אחוז מכלל הקולות הכשרים")
total_ax.set_ylabel("מפלגה")
total_ax.xaxis.set_major_formatter(PercentFormatter())
total_ax.grid(axis="x", alpha=0.25)
fig_total.tight_layout()
fig_total.canvas.layout.width = "72%"

party_legend_rows = "".join(
    f"<tr>"
    f"<td style='padding:3px 8px'><span style='display:inline-block;width:12px;height:12px;background:{party_colors.get(party, party_fallback_color)}'></span></td>"
    f"<th style='padding:3px 8px;text-align:right'>{party}</th>"
    f"<td style='padding:3px 8px;text-align:right'>{party_names.get(party, party)}</td>"
    f"</tr>"
    for party in reversed(party_percent.index)
)
party_legend = HTML(
    value=(
        "<div dir='rtl'><h3>אותיות המפלגות</h3>"
        f"<table style='border-collapse:collapse'>{party_legend_rows}</table></div>"
    ),
    layout=Layout(width="28%", max_height="650px", overflow="auto"),
)
results_summary = HBox(
    [fig_total.canvas, party_legend],
    layout=Layout(width="90vw", align_items="flex-start"),
)
results_summary
